# Pinpoint · WordNet Edition (Juego en Consola / Colab)

Juego de adivinanza léxica basado en relaciones semánticas de **WordNet** (hiperónimos, hipónimos, sinónimos) y normalización morfológica con **SnowballStemmer**.

### Instrucciones:
1. Ejecuta cada celda en orden secuencial.
2. En la última celda, interactúa ingresando el idioma de las pistas (`spa`, `fra`, `eng`), el idioma de la respuesta y el número de rondas.

In [1]:
# Celda 1: Descarga de recursos e inicialización de stemmers
import nltk
import unicodedata
import random as rd
from nltk.corpus import wordnet as wn
from nltk.stem.snowball import SnowballStemmer

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

stemmer_fr = SnowballStemmer("french")
stemmer_en = SnowballStemmer("english")
stemmer_es = SnowballStemmer("spanish")

print("Dependencias y recursos cargados correctamente.")

/home/maximilien-godin/Documents/UADY/projet_nlp/Pinpoint_Game/venv/lib/python3.12/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/maximilien-godin/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):


Dependencias y recursos cargados correctamente.


In [2]:
# Celda 2: Módulo de normalización y stemming (Lematización)
def normalize(texte: str) -> str:
    texte = texte.lower().strip()
    texte = unicodedata.normalize('NFD', texte)
    texte = ''.join(c for c in texte if unicodedata.category(c) != 'Mn')
    return texte

def stemmer_word(word: str, lang: str) -> str:
    original = normalize(word)
    if lang == "fra":
        stemmed = stemmer_fr.stem(original)
    elif lang == "spa":
        stemmed = stemmer_es.stem(original)
    elif lang == "eng":
        stemmed = stemmer_en.stem(original)
    else:
        return original

    # Regla de seguridad para tolerar plurales que el stemmer no reduzca
    if stemmed == original and original.endswith("s") and len(original) > 3:
        stemmed = original[:-1]
    return stemmed

def is_same_word(proposition: str, answer: str, lang: str) -> bool:
    return stemmer_word(proposition, lang) == stemmer_word(answer, lang)

In [3]:
# Celda 3: Módulo de pistas léxicas y selección con WordNet
MOTS_ANGLAIS = [
    "guitar", "violin", "piano", "drum", "trumpet", "flute", "harp", "saxophone",
    "lawyer", "teacher", "doctor", "surgeon", "farmer", "engineer", "pilot", "nurse",
    "firefighter", "police_officer", "chef", "dentist", "architect", "plumber",
    "electrician", "mechanic", "astronaut", "scientist", "painter", "photographer",
    "elephant", "penguin", "kangaroo", "dolphin", "giraffe", "zebra", "tiger", "lion",
    "panda", "koala", "octopus", "eagle", "owl", "butterfly", "spider", "crocodile",
    "umbrella", "backpack", "camera", "telescope", "microscope", "compass", "map",
    "calculator", "keyboard", "mirror", "candle", "lantern", "basket", "ladder",
    "hospital", "airport", "library", "museum", "cathedral", "castle", "lighthouse",
    "stadium", "theater", "bakery", "pharmacy", "supermarket", "restaurant", "hotel",
    "bicycle", "helicopter", "sailboat", "submarine", "train", "airplane", "rocket",
    "motorcycle", "canoe", "tractor", "ambulance",
    "waterfall", "volcano", "mountain", "desert", "forest", "glacier", "island",
    "river", "cave", "cliff", "valley", "beach", "lake",
    "violinist", "carpenter", "blacksmith", "beekeeper",
]

MOTS_ESPAGNOL = [
    "guitarra", "violín", "piano", "tambor", "trompeta", "flauta", "arpa", "saxofón",
    "abogado", "profesor", "médico", "cirujano", "agricultor", "ingeniero", "piloto",
    "enfermera", "bombero", "policía", "cocinero", "dentista", "arquitecto", "fontanero",
    "electricista", "mecánico", "astronauta", "científico", "pintor", "fotógrafo",
    "elefante", "pingüino", "canguro", "delfín", "jirafa", "cebra", "tigre", "león",
    "panda", "koala", "pulpo", "águila", "búho", "mariposa", "araña", "cocodrilo",
    "paraguas", "mochila", "cámara", "telescopio", "microscopio", "brújula", "mapa",
    "calculadora", "teclado", "espejo", "vela", "linterna", "cesta", "escalera",
    "hospital", "aeropuerto", "biblioteca", "museo", "catedral", "castillo", "faro",
    "estadio", "teatro", "panadería", "farmacia", "supermercado", "restaurante", "hotel",
    "bicicleta", "helicóptero", "velero", "submarino", "tren", "avión", "cohete",
    "motocicleta", "canoa", "tractor", "ambulancia",
    "cascada", "volcán", "montaña", "desierto", "bosque", "glaciar", "isla",
    "río", "cueva", "acantilado", "valle", "playa", "lago",
    "violinista", "carpintero", "herrero", "apicultor",
]

MOTS_FRANCAIS = [
    "guitare", "violon", "piano", "tambour", "trompette", "flûte", "harpe", "saxophone",
    "avocat", "professeur", "médecin", "chirurgien", "agriculteur", "ingénieur", "pilote",
    "infirmière", "pompier", "policier", "cuisinier", "dentiste", "architecte", "plombier",
    "électricien", "mécanicien", "astronaute", "scientifique", "peintre", "photographe",
    "éléphant", "pingouin", "kangourou", "dauphin", "girafe", "zèbre", "tigre", "lion",
    "panda", "koala", "poulpe", "aigle", "hibou", "papillon", "araignée", "crocodile",
    "parapluie", "sac_à_dos", "appareil_photo", "télescope", "microscope", "boussole",
    "carte", "calculatrice", "clavier", "miroir", "bougie", "lanterne", "panier", "échelle",
    "hôpital", "aéroport", "bibliothèque", "musée", "cathédrale", "château", "phare",
    "stade", "théâtre", "boulangerie", "pharmacie", "supermarché", "restaurant", "hôtel",
    "vélo", "hélicoptère", "voilier", "sous-marin", "train", "avion", "fusée",
    "moto", "canoë", "tracteur", "ambulance",
    "cascade", "volcan", "montagne", "désert", "forêt", "glacier", "île",
    "rivière", "grotte", "falaise", "vallée", "plage", "lac",
    "violiniste", "charpentier", "forgeron", "apiculteur",
]

def nombre_seguro(synset, idioma):
    if idioma == "eng":
        lemas = synset.lemmas()
    else:
        lemas = synset.lemmas(lang=idioma)
    if not lemas:
        return None
    nombre = lemas[0].name().replace("_", " ")
    return nombre.split("|")[0]

def _banco(idioma):
    if idioma == "spa":
        return MOTS_ESPAGNOL
    elif idioma == "fra":
        return MOTS_FRANCAIS
    return MOTS_ANGLAIS

def obtener_synset_por_indice(indice, idioma):
    palabra_en = MOTS_ANGLAIS[indice]
    synsets_en = wn.synsets(palabra_en, pos=wn.NOUN)
    if not synsets_en:
        return None
    if idioma == "eng":
        return synsets_en[0]

    palabra_cible = _banco(idioma)[indice]
    synsets_cible = set(wn.synsets(palabra_cible, lang=idioma, pos=wn.NOUN))
    if not synsets_cible:
        return None

    for s in synsets_en:
        if s in synsets_cible:
            return s
    return next(iter(synsets_cible))

def crear_palabras(idioma_i, max_intentos=500):
    n = len(MOTS_ANGLAIS)
    for _ in range(max_intentos):
        indice = rd.randrange(n)
        s = obtener_synset_por_indice(indice, idioma_i)
        if s is None:
            continue
        if (len(s.hypernyms()) > 0
                and len(s.hyponyms()) > 1
                and nombre_seguro(s, idioma_i) is not None):
            return indice
    raise RuntimeError("No se encontró una palabra válida en el banco.")

def obtener_indicios(indice, idioma_i):
    synset = obtener_synset_por_indice(indice, idioma_i)
    palabra = _banco(idioma_i)[indice]
    indicios = []
    origen = []

    hyperonymos = synset.hypernyms()
    hyponyms = synset.hyponyms()

    # 1. Hiperónimos
    candidatos_hiper = list(hyperonymos)
    if hyperonymos:
        candidatos_hiper += hyperonymos[0].hypernyms()

    for h in candidatos_hiper:
        if len(indicios) >= 2:
            break
        nombre = nombre_seguro(h, idioma_i)
        if nombre and nombre.lower() != palabra.replace("_", " ").lower() and nombre not in indicios:
            indicios.append(nombre)
            origen.append("hiperónimo")

    # 2. Hipónimos
    rd.shuffle(hyponyms)
    for h in hyponyms:
        if len(indicios) >= 4:
            break
        nombre = nombre_seguro(h, idioma_i)
        if nombre and nombre.lower() != palabra.replace("_", " ").lower() and nombre not in indicios:
            indicios.append(nombre)
            origen.append("hipónimo")

    # 3. Sinónimos
    if idioma_i == "eng":
        lemas = synset.lemmas()
    else:
        lemas = synset.lemmas(lang=idioma_i)

    sinonimos = [l.name().replace("_", " ").split("|")[0] for l in lemas]
    sinonimos = [s for s in sinonimos if s.lower() != palabra.replace("_", " ").lower()]
    rd.shuffle(sinonimos)
    for s in sinonimos:
        if len(indicios) >= 5:
            break
        if s not in indicios:
            indicios.append(s)
            origen.append("sinónimo")
            return indicios[:5], origen[:5]
def generar_partida(idioma_i, idioma_preg, max_intentos=100):
    for _ in range(max_intentos):
        indice = crear_palabras(idioma_i)
        indicios, origen = obtener_indicios(indice, idioma_i)
        if len(indicios) == 5:
            palabra_pista = _banco(idioma_i)[indice]
            palabra_respuesta = _banco(idioma_preg)[indice]
            return palabra_pista, indicios, origen, palabra_respuesta

    palabra_pista = _banco(idioma_i)[indice]
    palabra_respuesta = _banco(idioma_preg)[indice]
    return palabra_pista, indicios, origen, palabra_respuesta

## Lógica de la partida (clase `Game`)

Esta celda define la **clase `Game`**, que contiene toda la lógica del juego: las rondas, las pistas, el puntaje y la comprobación de las respuestas. No lee nada por consola ni dibuja ninguna interfaz, así que la pueden usar tanto la versión por consola como la versión con `ipywidgets`.

### Constantes

- `IDIOMAS_VALIDOS`: los códigos de idioma aceptados (`eng`, `fra`, `spa`).
- `NOMBRES_IDIOMA`: traduce cada código a su nombre en español, para mostrarlo en los mensajes.

### Clase `Game`

**`__init__`**: recibe el número de rondas, el idioma de las pistas, el idioma de la respuesta y la función `generar_partida_fn`. Prepara todas las rondas de antemano: llama a `generar_partida_fn` hasta tener `round_count` rondas válidas y descarta las que vengan incompletas. Cada ronda es un diccionario con:

| Clave | Contenido |
|---|---|
| `palabra_pista` | la palabra de partida |
| `indicios` | la lista de pistas |
| `origen` | el origen de cada pista (hiperónimo, hipónimo o sinónimo) |
| `palabra_respuesta` | la respuesta esperada |

También inicializa el estado de la partida:

| Atributo | Significado |
|---|---|
| `r` | índice de la ronda actual |
| `clues` | índice de la pista actual (0 a 4) |
| `score` | puntaje acumulado |
| `round_finished` | indica si la ronda actual ya terminó |
| `n_round` | número total de rondas |

**`is_last()`**: devuelve `True` si la ronda actual es la última.

**`answer(text)`**: comprueba la respuesta del jugador y devuelve un código con el resultado:

| Código devuelto | Situación |
|---|---|
| `"Round_finished"` | la ronda ya había terminado |
| `"Empty"` | la respuesta está vacía tras normalizarla con `normalize()` |
| `"words_found"` | respuesta correcta según `is_same_word()`; suma `6 - clues` puntos |
| `"wrong"` | respuesta incorrecta; pasa a la pista siguiente |
| `"out_of_clues"` | fallo con la última pista; la ronda termina sin puntos |

Como el puntaje es `6 - clues`, acertar con la primera pista da 6 puntos y con la quinta da 2. Cuantas menos pistas se necesiten, más puntos se ganan.

**`next_round()`**: pasa a la ronda siguiente y reinicia `clues` y `round_finished`. Devuelve `False` si la ronda actual no ha terminado o si ya era la última, lo que sirve para saber cuándo acaba la partida.

### Dependencias

`answer()` usa `normalize()` e `is_same_word()`, definidas en la celda 2 (normalización y stemming). `generar_partida` viene de la celda 3 (pistas con WordNet) y se pasa a `Game` como argumento. Hay que ejecutar esas celdas antes que esta.

In [ ]:
# Celda 4: Lógica de la partida e interfaz de usuario por consola
IDIOMAS_VALIDOS = ("eng", "fra", "spa")
NOMBRES_IDIOMA = {"eng": "inglés", "fra": "francés", "spa": "español"}

class Game:
    def __init__(self, round_count, langage_pistas, langage_respuesta, generar_partida_fn):
        self.langage_pistas = langage_pistas
        self.langage_respuesta = langage_respuesta
        self.rounds = []
        while len(self.rounds) < round_count:
            palabra_pista, indicios, origen, palabra_respuesta = generar_partida_fn(
                langage_pistas, langage_respuesta
            )
            if palabra_pista and indicios and palabra_respuesta:
                self.rounds.append({
                    "palabra_pista": palabra_pista,
                    "indicios": indicios,
                    "origen": origen,
                    "palabra_respuesta": palabra_respuesta,
                })
        self.r = 0
        self.clues = 0
        self.score = 0
        self.round_finished = False
        self.n_round = round_count

    def is_last(self):
        return self.r == (self.n_round - 1)

    def answer(self, text):
        if self.round_finished:
            return "Round_finished"
        word_normalised = normalize(text)
        if not word_normalised:
            return "Empty"
        respuesta_esperada = self.rounds[self.r]["palabra_respuesta"]
        if is_same_word(text, respuesta_esperada, self.langage_respuesta):
            self.score += 6 - self.clues
            self.round_finished = True
            return "words_found"
        if self.clues < 4:
            self.clues += 1
            return "wrong"
        self.round_finished = True
        return "out_of_clues"

    def next_round(self):
        if not self.round_finished or self.is_last():
            return False
        self.r += 1
        self.clues = 0
        self.round_finished = False
        return True







=== Pinpoint · WordNet Edition ===

Por favor, ingrese un número entero positivo.

Pistas en francés, respuestas en francés.
Generando partida...

=== Ronda 1/3 | Puntaje: 0 ===
Pista 1 [hiperónimo]: créateur
  Incorrecto, aquí tienes otra pista...
Pista 2 [hiperónimo]: être humain
  Incorrecto, aquí tienes otra pista...
Pista 3 [hipónimo]: forestier
  Incorrecto, aquí tienes otra pista...
Pista 4 [hipónimo]: apiculteur
  ¡Correcto! La respuesta era 'agriculteur'. Puntaje: 3

=== Ronda 2/3 | Puntaje: 3 ===
Pista 1 [hiperónimo]: flore
  Incorrecto, aquí tienes otra pista...
Pista 2 [hiperónimo]: agrégation
  Incorrecto, aquí tienes otra pista...
Pista 3 [hipónimo]: bosquet
  ¡Correcto! La respuesta era 'forêt'. Puntaje: 7

=== Ronda 3/3 | Puntaje: 7 ===
Pista 1 [hiperónimo]: oiseau de mer
  Incorrecto, aquí tienes otra pista...
Pista 2 [hipónimo]: jackass penguin
  Incorrecto, aquí tienes otra pista...
Pista 3 [hipónimo]: manchot adélie
  Incorrecto, aquí tienes otra pista...
Pista 4 [h

## Para jugar con Interfaz gráfica del juego (ipywidgets)

Esta celda reemplaza la versión por consola por una **interfaz interactiva** que funciona directamente en Google Colab. La lógica del juego (clase `Game`, pistas de WordNet, normalización) no cambia; solo cambia la forma de interactuar con ella.

### Cómo funciona

1. **Configuración**: el jugador elige el idioma de las pistas, el idioma de la respuesta y el número de rondas con menús desplegables y un control deslizante. El botón *Empezar la partida* crea un objeto `Game` con esos parámetros.
2. **Ronda**: se muestra la pista actual junto con su origen (hiperónimo, hipónimo o sinónimo) y la puntuación. La respuesta se envía con el botón *Validar* o con la tecla Enter.
3. **Resultado**: si la respuesta es incorrecta, aparece una nueva pista; si es correcta o se agotan las pistas, se muestra el resultado y, tras una pausa de 3 segundos, comienza la siguiente ronda. Cuantas menos pistas se necesiten, más puntos se ganan.
4. **Final**: al terminar la última ronda se muestra la puntuación final. El botón *Abandonar* cancela la partida en cualquier momento.

### Componentes principales

| Elemento | Función |
|---|---|
| `widgets.Dropdown`, `IntSlider`, `Text`, `Button` | Reemplazan los `input()` de la consola |
| `widgets.Output` | Zona donde se muestran las pistas y los mensajes |
| `mostrar_texto()` | Muestra un mensaje centrado usando HTML |
| `mostrar_ronda()` | Borra la zona de salida y muestra la ronda y la pista actuales |
| `al_hacer_clic_empezar()` | Crea la partida al pulsar *Empezar la partida* |
| `procesar_respuesta()` | Comprueba la respuesta, actualiza el estado y pasa a la siguiente pista o ronda |
| `al_hacer_clic_abandonar()` | Cancela la partida y muestra la puntuación |
| `widgets.Layout` | Centra los elementos y fija un ancho común de 360 px |

### Notas técnicas

- Cada botón tiene una función asociada (*callback*) que se ejecuta al hacer clic, en lugar de un bucle `while` que espera la entrada del usuario.
- `game_state` es un diccionario que guarda la partida en curso para que todas las funciones puedan acceder a ella.
- `time.sleep(3)` deja tiempo para leer el resultado antes de que `clear_output` limpie la pantalla.

In [14]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from IPython.display import HTML
import html
import time
#
box_layout = widgets.Layout(width="360px")          # largeur commune des champs
center_col = widgets.Layout(align_items="center")   # centre les enfants d'un VBox
center_row = widgets.Layout(justify_content="center")  # centre les enfants d'un HBox

# --- Widgets de configuration ---
lang_pistas_w = widgets.Dropdown(
    options=[("Inglés", "eng"), ("Francés", "fra"), ("Español", "spa")],
    value="fra", description="Pistas :", layout = box_layout
)
lang_respuesta_w = widgets.Dropdown(
    options=[("Inglés", "eng"), ("Francés", "fra"), ("Español", "spa")],
    value="fra", description="Respuesta :", layout=box_layout
)
n_rounds_w = widgets.IntSlider(value=3, min=1, max=10, step=1, description="Rondas :", layout=box_layout)
start_button = widgets.Button(description="Empezar la partida", button_style="success", icon="play")
config_box = widgets.VBox([lang_pistas_w, lang_respuesta_w, n_rounds_w, start_button], layout = center_col)

# --- Widgets de jeu ---
answer_input = widgets.Text(placeholder="Tu respuesta...", description="Respuesta :", layout=box_layout)
answer_button = widgets.Button(description="Validar", button_style="primary", icon="check")
quit_button = widgets.Button(description="Abandonar", button_style="danger", icon="stop")
answer_box = widgets.HBox([answer_input, answer_button, quit_button], layout=center_row)

output = widgets.Output(layout=center_col)
game_state = {"game": None}

def mostrar_texto(msg, gras=False):
    contenu = html.escape(msg)
    if gras:
        contenu = f"<b>{contenu}</b>"
    display(HTML(f"<div style='text-align:center; font-family:monospace; font-size:15px'>{contenu}</div>"))

def mostrar_ronda():
    game = game_state["game"]
    with output:
        clear_output(wait=True)
        mostrar_texto(f"=== Ronda {game.r + 1}/{game.n_round} | Puntuación : {game.score} ===")
        clue = game.rounds[game.r]["indicios"][game.clues]
        origen = game.rounds[game.r]["origen"][game.clues]
        mostrar_texto(f"Pista {game.clues + 1} [{origen}] : {clue}")
        display(answer_box)

def al_hacer_clic_empezar(b):
    with output:
        clear_output(wait=True)
        mostrar_texto ("Generando la partida...")
    game_state["game"] = Game(n_rounds_w.value, lang_pistas_w.value, lang_respuesta_w.value, generar_partida)
    mostrar_ronda()

def procesar_respuesta(_):
    game = game_state["game"]
    if game is None:
        return
    texte = answer_input.value.strip()
    answer_input.value = ""
    resultado = game.answer(texte)

    if resultado == "Empty":
        with output:
            mostrar_texto("Escribe una respuesta.")
        return

    with output:
        if resultado == "wrong":
            mostrar_texto("Incorrecto, aquí tienes otra pista...")
        elif resultado == "words_found":
            mostrar_texto(f"¡Correcto! La respuesta era '{game.rounds[game.r]['palabra_respuesta']}'. Puntos : {game.score}\n")
        elif resultado == "out_of_clues":
            mostrar_texto(f"No quedan más pistas. La respuesta correcta era '{game.rounds[game.r]['palabra_respuesta']}'.\n")

    if game.round_finished:
        if not game.next_round():
            with output:
                clear_output(wait=True)
                mostrar_texto(f"=== Fin de la partida. Puntos finales : {game.score} ===")
            return

    time.sleep(3)
    mostrar_ronda()

def al_hacer_clic_abandonar(b):
    game = game_state["game"]
    with output:
        clear_output(wait=True)
        mostrar_texto(f"Partida cancelada. Puntos finales: {game.score if game else 0}")

start_button.on_click(al_hacer_clic_empezar)
answer_button.on_click(procesar_respuesta)
answer_input.on_submit(procesar_respuesta)  # valider avec Entrée
quit_button.on_click(al_hacer_clic_abandonar)

app = widgets.VBox([config_box, output], layout=widgets.Layout(align_items="center", width="100%"))
display(app)

/tmp/ipykernel_5482/4207124373.py:95: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  answer_input.on_submit(procesar_respuesta)  # valider avec Entrée


## Para jugar en la consola

Esta celda contiene la función `play_console()`, que permite jugar a Pinpoint escribiendo en la consola con `input()` y `print()`. Es la versión sin interfaz gráfica: la lógica del juego (clase `Game`, pistas de WordNet, normalización) es la misma que usa la versión con `ipywidgets`.

### Cómo funciona

1. **Configuración**: se piden por consola el idioma de las pistas y el idioma de la respuesta esperada con `pedir_idioma()`, y el número de rondas con `pedir_numero_rondas()`.
2. **Creación de la partida**: se genera un objeto `Game` con esos parámetros, que prepara todas las rondas de antemano.
3. **Bucle de rondas**: en cada ronda se muestra una pista con su origen (hiperónimo, hipónimo o sinónimo) y el jugador escribe su respuesta. `game.answer()` devuelve el resultado:
   - `"Empty"`: la respuesta está vacía, se vuelve a pedir.
   - `"wrong"`: respuesta incorrecta, se muestra la siguiente pista.
   - `"words_found"`: respuesta correcta, se suman puntos (cuantas menos pistas se usen, más puntos).
   - `"out_of_clues"`: se acabaron las pistas, se revela la respuesta.
4. **Fin**: cuando `game.next_round()` devuelve `False`, la partida termina y se muestra el puntaje final. Escribir `exit` en cualquier momento cancela la partida.

### Funciones de entrada

- **`pedir_idioma(mensaje)`**: muestra el mensaje con `input()` y repite la pregunta hasta que el jugador escribe un código válido (`eng`, `fra` o `spa`). Si se equivoca, avisa con "Idioma no válido". Se llama dos veces: una para el idioma de las pistas y otra para el de la respuesta.
- **`pedir_numero_rondas()`**: pregunta cuántas rondas se quieren jugar y repite la pregunta hasta que el jugador escribe un número entero positivo. Devuelve ese número como `int`.

Las dos usan un bucle `while` que solo termina cuando la entrada es válida, así `Game` siempre recibe parámetros correctos.

### Estructura del código

| Elemento | Función |
|---|---|
| `pedir_idioma()` | Lee y valida el idioma de las pistas y el de la respuesta |
| `pedir_numero_rondas()` | Lee y valida el número de rondas |
| `while True` (exterior) | Recorre las rondas de la partida |
| `while not game.round_finished` (interior) | Muestra pistas y pide respuestas hasta terminar la ronda |
| `input()` | Lee la respuesta del jugador |
| `return` | Sale de la función si el jugador escribe `exit` |
| `break` | Sale del bucle exterior cuando no quedan más rondas |

### Diferencia con la versión gráfica

Aquí el programa **espera** la respuesta con `input()` dentro de un bucle. En la versión con `ipywidgets` no hay bucle: cada botón tiene una función asociada (*callback*) que se ejecuta al hacer clic, y los menús desplegables y el control deslizante sustituyen a `pedir_idioma()` y `pedir_numero_rondas()`.

In [ ]:
# Iniciar juego
def pedir_idioma(mensaje):
    idioma = ""
    while idioma not in IDIOMAS_VALIDOS:
        idioma = input(mensaje).strip().lower()
        if idioma not in IDIOMAS_VALIDOS:
            print("Idioma no válido. Ingrese 'eng', 'fra' o 'spa'.")
    return idioma

def pedir_numero_rondas():
    while True:
        texto = input("¿Cuántas rondas deseas jugar? (ej: 3): ").strip()
        if texto.isdigit() and int(texto) > 0:
            return int(texto)
        print("Por favor, ingrese un número entero positivo.")
        
def play_console():
    print("=== Pinpoint · WordNet Edition ===\n")
    lang_pistas = pedir_idioma("Idioma de las pistas (eng, fra, spa): ")
    lang_respuesta = pedir_idioma("Idioma de la respuesta esperada (eng, fra, spa): ")
    n_round = pedir_numero_rondas()

    print(f"\nPistas en {NOMBRES_IDIOMA[lang_pistas]}, respuestas en {NOMBRES_IDIOMA[lang_respuesta]}.")
    print("Generando partida...\n")
    game = Game(n_round, lang_pistas, lang_respuesta, generar_partida)

    while True:
        print(f"=== Ronda {game.r + 1}/{game.n_round} | Puntaje: {game.score} ===")
        while not game.round_finished:
            clue = game.rounds[game.r]["indicios"][game.clues]
            origen = game.rounds[game.r]["origen"][game.clues]
            print(f"Pista {game.clues + 1} [{origen}]: {clue}")
            text = input("  Tu respuesta (o 'exit' para salir): ").strip()
            if text.lower() == "exit":
                print(f"\nPartida cancelada. Puntaje final: {game.score}")
                return
            resultado = game.answer(text)
            if resultado == "Empty":
                print("  Escribe una respuesta.")
            elif resultado == "wrong":
                print("  Incorrecto, aquí tienes otra pista...")
            elif resultado == "words_found":
                print(f"  ¡Correcto! La respuesta era '{game.rounds[game.r]['palabra_respuesta']}'. Puntaje: {game.score}\n")
            elif resultado == "out_of_clues":
                print(f"  Sin pistas. La respuesta correcta era '{game.rounds[game.r]['palabra_respuesta']}'.\n")
        if not game.next_round():
            break
    print(f"=== Fin de la partida. Puntaje final: {game.score} ===")
play_console()